## Puck 06 — unperturbed NMF sanity check (k=15)

Load Puck 06, split into top / bottom halves (same Y-midpoint split as B.1),
run NMF with k=15 on both halves jointly, and visualise each factor spatially.  
Goal: confirm that NMF captures the known brain-region annotation (`TopStruct`) in
the unperturbed data before any case/control perturbation is introduced.

In [ ]:
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 1

RAND_SEED = 28
N_FACTORS = 15
DATA_DIR  = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/All_Pucks_h5ad')

np.random.seed(RAND_SEED)

## 1. Load Puck 06

In [ ]:
adata06 = ad.read_h5ad(DATA_DIR / 'Puck_Num_06.h5ad')
adata06 = adata06[adata06.obs['IsOutsideCCF'] == 0].copy()
print(f'Tissue beads: {adata06.n_obs:,}')
print(adata06)

In [ ]:
def plot_puck(adata, title='', color='TopStruct', point_size=0.8):
    obs = adata.obs
    cats = sorted(obs[color].astype(str).unique(), key=lambda x: (x == 'NA', x))
    cmap = plt.colormaps['tab20'].resampled(len(cats))
    col_dict = {c: cmap(i) for i, c in enumerate(cats)}
    col_dict['NA'] = (0.82, 0.82, 0.82, 1.0)
    x = obs['Raw_Slideseq_X'].values.astype(float)
    y = obs['Raw_Slideseq_Y'].values.astype(float)
    c = [col_dict[str(v)] for v in obs[color]]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x, y, c=c, s=point_size, linewidths=0, edgecolors='none', rasterized=True)
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    ax.set_title(f'{title}  ({color}, n={len(obs):,})', fontsize=10)
    patches = [mpatches.Patch(color=col_dict[c], label=c) for c in cats]
    ax.legend(handles=patches, fontsize=7, loc='lower right', frameon=False, markerscale=4)
    plt.tight_layout()
    plt.show()

## 2. Split at Y midpoint → top & bottom

In [ ]:
# Find the natural split point in puck 06 by looking at the Y-axis bead density
y_all = adata06.obs['Raw_Slideseq_Y'].values.astype(float)

print(min(y_all), max(y_all))
split_point = min(y_all) + (max(y_all) - min(y_all)) / 2
print(f'Natural split point in Y for puck 06: {split_point:.1f}')
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(y_all, bins=100, color='steelblue', edgecolor='none')
ax.axvline(split_point, color='red', linestyle='--', label=f'split at Y={split_point:.1f}')
ax.set_xlabel('Raw_Slideseq_Y')
ax.set_ylabel('Bead count')
ax.set_title('Puck 06 — Y coordinate distribution (two sections visible as two peaks)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
adata_top = adata06[y_all <= split_point].copy()
adata_bot = adata06[y_all >  split_point].copy()

adata_top.obs['half'] = 'top'
adata_bot.obs['half'] = 'bottom'

print(f'Top half:    {adata_top.n_obs:,} beads')
print(f'Bottom half: {adata_bot.n_obs:,} beads')

# Flip bottom vertically so it mirrors the top (same as B.1)
y_bot = adata_bot.obs['Raw_Slideseq_Y'].values.astype(float)
adata_bot.obs['Raw_Slideseq_Y'] = y_bot.max() + y_bot.min() - y_bot

plot_puck(adata_top, 'Puck 06 — top')
plot_puck(adata_bot, 'Puck 06 — bottom (y-flipped)')

In [ ]:
plot_puck(adata_top, 'Puck 06 — top', color='DeepCCF')
plot_puck(adata_bot, 'Puck 06 — bottom (y-flipped)', color='DeepCCF')

## 3. Concatenate & preprocess

In [ ]:
adata = ad.concat(
    [adata_top, adata_bot],
    join='inner', merge='first',
    label='half', keys=['top', 'bottom'], index_unique='-'
)
# Restore per-bead columns lost during concat
adata.obs['half'] = np.concatenate([
    adata_top.obs['half'].values,
    adata_bot.obs['half'].values
])
print(adata)
print(adata.obs['half'].value_counts())

In [ ]:
adata.layers['counts'] = adata.X.copy()
print(f'Before gene filter: {adata.shape}')

min_cells = max(1, adata.n_obs // 1000)
n_expr    = np.array((adata.X > 0).sum(axis=0)).flatten()
adata     = adata[:, n_expr >= min_cells].copy()
print(f'After gene filter (min_cells={min_cells}): {adata.shape}')

# HVG selection: compute on a temporary lognorm copy, then apply mask to raw counts
# (seurat_v3 requires skmisc which is not installed; seurat/cell_ranger expect lognorm)
_tmp = adata.copy()
sc.pp.normalize_total(_tmp, target_sum=1e4)
sc.pp.log1p(_tmp)
sc.pp.highly_variable_genes(_tmp, n_top_genes=2000, flavor='seurat', subset=False)
adata.var['highly_variable'] = _tmp.var['highly_variable']
del _tmp
print(f'HVGs: {adata.var["highly_variable"].sum()}')

adata = adata[:, adata.var['highly_variable']].copy()
print(f'After HVG subset: {adata.shape}')

UNIT_VARIANCE_SCALING = False
# Unit-variance scaling without centering — preserves non-negativity for NMF
# adata.X is still raw counts at this point; no log-normalisation applied
if UNIT_VARIANCE_SCALING:
    sc.pp.scale(adata, zero_center=False)

## 4. NMF k=15

In [ ]:
X_nmf = adata.X
if sp.issparse(X_nmf):
    X_nmf = X_nmf.toarray()
X_nmf = X_nmf.astype(np.float32)

nmf_model = NMF(
    n_components=N_FACTORS,
    init='nndsvda',
    random_state=RAND_SEED,
    max_iter=1000,
    solver='cd',
)
W = nmf_model.fit_transform(X_nmf)
H = nmf_model.components_

adata.obsm['X_nmf'] = W
adata.uns['nmf_components'] = H

print(f'NMF W: {W.shape}')
print(f'Reconstruction error: {nmf_model.reconstruction_err_:.4f}')

## 5. Visualise each NMF factor over both halves

Top row = top half, bottom row = bottom half (y-flipped).  
A factor that captures a brain region should show a coherent spatial pattern in both halves.

In [ ]:
top_idx = np.where(adata.obs['half'] == 'top')[0]
bot_idx = np.where(adata.obs['half'] == 'bottom')[0]

for fi in range(N_FACTORS):
    scores = W[:, fi]
    vmax   = np.percentile(scores, 99)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f'NMF Factor {fi + 1}', fontsize=12)

    for ax, idx, label in [
        (axes[0], top_idx, 'top'),
        (axes[1], bot_idx, 'bottom (y-flipped)'),
    ]:
        sub_obs = adata.obs.iloc[idx]
        x       = sub_obs['Raw_Slideseq_X'].values.astype(float)
        y_coord = sub_obs['Raw_Slideseq_Y'].values.astype(float)
        s       = scores[idx]
        sc_     = ax.scatter(x, y_coord, c=s, s=1, cmap='magma_r',
                             vmin=0, vmax=vmax,
                             linewidths=0, edgecolors='none', rasterized=True)
        plt.colorbar(sc_, ax=ax, fraction=0.03, pad=0.02)
        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'Puck 06 — {label}  ({len(idx):,} beads)', fontsize=9)

    plt.tight_layout()
    plt.show()

## 6. NMF factors vs TopStruct

Heatmap of median NMF score per `TopStruct` region — a spatially-specific factor should
have high median in one or two regions and near-zero elsewhere.

In [ ]:

nmf_df = pd.DataFrame(
    W,
    columns=[f'F{i+1}' for i in range(N_FACTORS)],
    index=adata.obs_names
)
nmf_df['DeepCCF'] = adata.obs['DeepCCF'].values

factor_cols = [f'F{i+1}' for i in range(N_FACTORS)]

median_by_struct = (
    nmf_df.groupby('DeepCCF')[factor_cols].median()
)
counts_per_region = nmf_df.groupby('DeepCCF').size().rename('n')

# Column-normalise each factor to [0, 1] so low-activation factors are still visible
mat = median_by_struct.values.copy()
col_max = mat.max(axis=0)
col_max[col_max == 0] = 1
mat_norm = mat / col_max

fig, ax = plt.subplots(figsize=(N_FACTORS * 0.9 + 2, len(median_by_struct) * 0.35 + 2))
im = ax.imshow(mat_norm, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02,
             label='median NMF score\n(column-normalised per factor)')
ax.set_xticks(range(N_FACTORS))
ax.set_xticklabels(factor_cols, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(median_by_struct)))
ax.set_yticklabels(median_by_struct.index, fontsize=7)
ax.set_title('Median NMF factor score per DeepCCF region', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
nmf_df['TopStruct'] = adata.obs['TopStruct'].values

median_by_topstruct   = nmf_df.groupby('TopStruct')[factor_cols].median()
counts_per_topstruct  = nmf_df.groupby('TopStruct').size().rename('n')

mat_ts     = median_by_topstruct.values.copy()
col_max_ts = mat_ts.max(axis=0)
col_max_ts[col_max_ts == 0] = 1
mat_ts_norm = mat_ts / col_max_ts

fig, ax = plt.subplots(figsize=(N_FACTORS * 0.9 + 2, len(median_by_topstruct) * 0.35 + 2))
im = ax.imshow(mat_ts_norm, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02,
             label='median NMF score\n(column-normalised per factor)')
ax.set_xticks(range(N_FACTORS))
ax.set_xticklabels(factor_cols, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(median_by_topstruct)))
ax.set_yticklabels(
    [f'{r}  (n={counts_per_topstruct[r]:,})' for r in median_by_topstruct.index],
    fontsize=8
)
ax.set_title('Median NMF factor score per TopStruct region', fontsize=11)
plt.tight_layout()
plt.show()


## 7. Top genes per factor

In [ ]:
gene_names = adata.var['features'].values if 'features' in adata.var.columns else adata.var_names.values

for fi in range(N_FACTORS):
    top10 = gene_names[np.argsort(H[fi])[::-1][:10]]
    print(f'F{fi+1:>2}: {list(top10)}')